# EDA — MovieLens 1M

**Goal**: Understand the data well enough to finalize the train/test split strategy, confirm whether feature engineering is needed, and anticipate problems.

**Files**
- `ratings.dat`: UserID, MovieID, Rating, Timestamp (1M rows)
- `users.dat`  : UserID, Gender, Age, Occupation, Zip-code (6,040 rows)
- `movies.dat` : MovieID, Title, Genres (3,883 rows)

**Key questions**
1. Basic statistics: how many users, items, ratings? What is the matrix density?
2. User activity: distribution of interactions per user — where is the long tail? How does 20% subsampling hurt the sparsest users?
3. Item popularity: power-law? How many items are never in the top-100 candidates?
4. Rating distribution: do we binarize (implicit) or keep as-is?
5. Temporal structure: are timestamps usable for leave-one-out split?
6. User demographics and movie genres: do we need them for the models?
7. ID gaps and data quality issues.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
DATA = Path('../data/raw/ml-1m')
FIG  = Path('../figures')
FIG.mkdir(exist_ok=True)

## 1. Load raw data

In [ ]:
ratings = pd.read_csv(
    DATA / 'ratings.dat', sep='::', engine='python', header=None,
    names=['user_id', 'movie_id', 'rating', 'timestamp'],
    dtype={'user_id': np.int32, 'movie_id': np.int32, 'rating': np.float32, 'timestamp': np.int64}
)

users = pd.read_csv(
    DATA / 'users.dat', sep='::', engine='python', header=None,
    names=['user_id', 'gender', 'age', 'occupation', 'zip'],
    dtype={'user_id': np.int32}
)

movies = pd.read_csv(
    DATA / 'movies.dat', sep='::', engine='python', header=None,
    names=['movie_id', 'title', 'genres'],
    encoding='latin-1',
    dtype={'movie_id': np.int32}
)

print(f'Ratings : {len(ratings):,}')
print(f'Users   : {len(users):,}')
print(f'Movies  : {len(movies):,}')
ratings.head()

## 2. Basic statistics

In [ ]:
n_users  = ratings['user_id'].nunique()
n_items  = ratings['movie_id'].nunique()
n_ratings = len(ratings)
density  = n_ratings / (n_users * n_items)

print(f'Unique users  : {n_users:,}')
print(f'Unique movies : {n_items:,}')
print(f'Ratings       : {n_ratings:,}')
print(f'Matrix density: {density:.4%}  ({density:.5f})')
print(f'Sparsity      : {1-density:.4%}')
print()
print('Rating value distribution:')
print(ratings['rating'].value_counts().sort_index())

In [ ]:
# Check for movie IDs in ratings that are NOT in movies.dat
missing_movies = set(ratings['movie_id'].unique()) - set(movies['movie_id'].unique())
print(f'Movie IDs in ratings but not in movies.dat: {len(missing_movies)}')
print(f'Movie ID range in ratings.dat: {ratings["movie_id"].min()} – {ratings["movie_id"].max()}')
print(f'Movie ID range in movies.dat : {movies["movie_id"].min()} – {movies["movie_id"].max()}')

# Are all users in ratings also in users.dat?
missing_users = set(ratings['user_id'].unique()) - set(users['user_id'].unique())
print(f'User IDs in ratings but not in users.dat: {len(missing_users)}')

## 3. Rating distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw rating counts
vc = ratings['rating'].value_counts().sort_index()
axes[0].bar(vc.index, vc.values, color=sns.color_palette()[0], edgecolor='white')
axes[0].set_xlabel('Star Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Raw Rating Distribution')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# Cumulative proportion
axes[1].bar(vc.index, vc.values / vc.sum() * 100, color=sns.color_palette()[1], edgecolor='white')
axes[1].set_xlabel('Star Rating')
axes[1].set_ylabel('% of Ratings')
axes[1].set_title('Rating Distribution (%)')

plt.tight_layout()
plt.savefig(FIG / 'eda_rating_distribution.png', dpi=150)
plt.show()

print(f'Mean rating: {ratings["rating"].mean():.3f}')
print(f'Median     : {ratings["rating"].median():.1f}')
print()
print('NOTE: For implicit feedback models (MF, NCF, ranker) we treat any observed')
print('rating as a positive interaction and ignore the star value.')
print('Binarization: rating >= 1 → 1 (positive), unobserved → 0 (negative).')

## 4. User activity distribution — critical for the sparsity sweep

In [ ]:
user_counts = ratings.groupby('user_id').size().rename('n_ratings')

print('Per-user interaction count summary:')
print(user_counts.describe(percentiles=[.05, .1, .25, .5, .75, .9, .95]))
print(f'\nMin interactions (README claims ≥20): {user_counts.min()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram on linear scale
axes[0].hist(user_counts, bins=80, color=sns.color_palette()[2], edgecolor='white', linewidth=0.3)
axes[0].set_xlabel('Ratings per User')
axes[0].set_ylabel('Number of Users')
axes[0].set_title('User Activity Distribution (linear)')

# Histogram on log scale to see the tail
axes[1].hist(user_counts, bins=80, color=sns.color_palette()[3], edgecolor='white', linewidth=0.3)
axes[1].set_yscale('log')
axes[1].set_xlabel('Ratings per User')
axes[1].set_ylabel('Number of Users (log)')
axes[1].set_title('User Activity Distribution (log y)')

plt.tight_layout()
plt.savefig(FIG / 'eda_user_activity.png', dpi=150)
plt.show()

In [ ]:
# How many training interactions does each density level leave? 
# Leave-one-out: 1 interaction is held out for test, so training pool = n_ratings - 1.
# At density d, we subsample d * (n_ratings - 1) interactions for training.

print('Training interactions per user after leave-one-out + subsampling:')
print(f'{"Density":>10} | {"Min":>8} | {"Median":>8} | {"p10":>8} | {"% users < 5 train":>20}')
print('-' * 65)
for d in [1.0, 0.8, 0.6, 0.4, 0.2]:
    train_pool = (user_counts - 1)  # remove the held-out test item
    train_at_d = (train_pool * d).astype(int).clip(lower=1)  # at least 1 train interaction
    pct_lt5 = (train_at_d < 5).mean() * 100
    print(f'{d:>10.0%} | {train_at_d.min():>8} | {train_at_d.median():>8.0f} | {train_at_d.quantile(.1):>8.0f} | {pct_lt5:>20.1f}%')

print()
print('DECISION: At 20% density some users will have only 4 training interactions.')
print('This is intentional — it is the "cold start" end of the sparsity curve.')
print('No user filtering needed; all 6,040 users stay in at every density level.')

## 5. Item popularity distribution

In [ ]:
item_counts = ratings.groupby('movie_id').size().rename('n_ratings').sort_values(ascending=False)

print('Per-item rating count summary:')
print(item_counts.describe(percentiles=[.25, .5, .75, .9, .95, .99]))
print(f'\nMovies with only 1 rating : {(item_counts == 1).sum()}')
print(f'Movies with < 5 ratings  : {(item_counts < 5).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Rank-frequency (Zipf) plot
axes[0].loglog(range(1, len(item_counts)+1), item_counts.values, color=sns.color_palette()[4], linewidth=1.2)
axes[0].set_xlabel('Item Rank (by popularity)')
axes[0].set_ylabel('Number of Ratings (log)')
axes[0].set_title('Item Popularity — Rank-Frequency (log-log)')

# Coverage: top-k% of items account for x% of ratings
cumulative = item_counts.cumsum() / item_counts.sum()
axes[1].plot(np.arange(1, len(cumulative)+1) / len(cumulative) * 100, cumulative.values * 100,
             color=sns.color_palette()[5], linewidth=1.5)
axes[1].axhline(80, linestyle='--', color='gray', linewidth=0.8, label='80% of ratings')
axes[1].set_xlabel('Top X% of Items (by popularity)')
axes[1].set_ylabel('Cumulative % of Ratings')
axes[1].set_title('Item Popularity — Cumulative Coverage')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG / 'eda_item_popularity.png', dpi=150)
plt.show()

# Find the top-X% of items covering 80% of ratings
pct_items_80 = (cumulative <= 0.80).sum() / len(cumulative) * 100
print(f'Top {pct_items_80:.1f}% of items account for 80% of all ratings — strong power law.')

## 6. Temporal structure — leave-one-out feasibility

In [ ]:
import datetime

ts_min = pd.to_datetime(ratings['timestamp'].min(), unit='s')
ts_max = pd.to_datetime(ratings['timestamp'].max(), unit='s')
print(f'Timestamp range: {ts_min.date()} → {ts_max.date()}')

# Check: do all users have distinct timestamps for their last interaction?
last_per_user = ratings.sort_values('timestamp').groupby('user_id').last()
users_with_ties = ratings[ratings['user_id'].isin(last_per_user.index)].groupby('user_id').apply(
    lambda g: (g['timestamp'] == g['timestamp'].max()).sum() > 1
)
n_ties = users_with_ties.sum()
print(f'Users where the last timestamp is shared by >1 rating: {n_ties} ({n_ties/n_users:.1%})')
print()
print('DECISION: Leave-one-out = hold out each user\'s most recent interaction by timestamp.')
print('For ties (same timestamp on multiple items), break by taking the row with the highest')
print('original index. This is deterministic and matches He et al. (2017) protocol.')

In [ ]:
# Visualize rating activity over time
ratings['date'] = pd.to_datetime(ratings['timestamp'], unit='s').dt.to_period('M')
monthly = ratings.groupby('date').size()

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(len(monthly)), monthly.values, color=sns.color_palette()[0], width=0.8)
ax.set_xticks(range(0, len(monthly), 4))
ax.set_xticklabels([str(d) for d in monthly.index[::4]], rotation=45, ha='right')
ax.set_xlabel('Month')
ax.set_ylabel('Ratings')
ax.set_title('Rating Activity Over Time')
plt.tight_layout()
plt.savefig(FIG / 'eda_temporal.png', dpi=150)
plt.show()

## 7. User demographics

In [ ]:
age_map = {1: '<18', 18: '18-24', 25: '25-34', 35: '35-44', 45: '45-49', 50: '50-55', 56: '56+'}
users['age_label'] = users['age'].map(age_map)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Gender
g = users['gender'].value_counts()
axes[0].bar(g.index, g.values, color=[sns.color_palette()[0], sns.color_palette()[1]])
axes[0].set_title('Gender')
axes[0].set_ylabel('Users')

# Age
age_order = ['<18', '18-24', '25-34', '35-44', '45-49', '50-55', '56+']
a = users['age_label'].value_counts().reindex(age_order)
axes[1].bar(a.index, a.values, color=sns.color_palette()[2])
axes[1].set_title('Age Group')
axes[1].tick_params(axis='x', rotation=30)

# Occupation (top 10)
occ_map = {0:'other',1:'academic',2:'artist',3:'clerical',4:'student',5:'customer svc',
           6:'doctor',7:'executive',8:'farmer',9:'homemaker',10:'K-12 student',
           11:'lawyer',12:'programmer',13:'retired',14:'sales',15:'scientist',
           16:'self-employed',17:'engineer',18:'tradesman',19:'unemployed',20:'writer'}
users['occ_label'] = users['occupation'].map(occ_map)
top_occ = users['occ_label'].value_counts().head(10)
axes[2].barh(top_occ.index[::-1], top_occ.values[::-1], color=sns.color_palette()[4])
axes[2].set_title('Top 10 Occupations')
axes[2].set_xlabel('Users')

plt.tight_layout()
plt.savefig(FIG / 'eda_demographics.png', dpi=150)
plt.show()

## 8. Movie genres

In [ ]:
# Each movie can have multiple genres (pipe-separated)
genre_series = movies['genres'].str.split('|').explode()
genre_counts = genre_series.value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(genre_counts.index[::-1], genre_counts.values[::-1], color=sns.color_palette()[3])
ax.set_xlabel('Number of Movies')
ax.set_title('Genre Distribution (multi-label)')
plt.tight_layout()
plt.savefig(FIG / 'eda_genres.png', dpi=150)
plt.show()

print(f'Movies with multiple genres: {(movies["genres"].str.contains("|", regex=False)).sum():,} / {len(movies):,}')

## 9. Dataset decisions — train / val / test split

In [ ]:
print("""
=== SPLIT STRATEGY (He et al. 2017 leave-one-out protocol) ===

For each user:
  - Sort their interactions by timestamp.
  - TEST  : last interaction (most recent).
  - VAL   : second-to-last interaction.
  - TRAIN : all remaining interactions (n_ratings - 2).

At each sparsity level d in {1.0, 0.8, 0.6, 0.4, 0.2}:
  - Subsample d * |TRAIN_user| interactions (rounded down, min=1).
  - VAL and TEST items are never subsampled — they always stay.

Evaluation:
  - For each user, sample 99 items the user has NOT rated as negatives.
  - Rank the test item among the 100 (1 positive + 99 negatives).
  - Compute NDCG@10 and HR@10 over all users.
  - This protocol is deterministic once we fix the negative sample seed.

=== FEATURE ENGINEERING DECISION ===

MF and NCF: NO feature engineering needed.
  - Input is purely (user_id, item_id) pairs.
  - Users and movies get contiguous 0-indexed IDs for embedding lookup.
  - Ratings are binarized: any observed rating → 1 (positive).

MF + Ranker: NO feature engineering for the core experiment.
  - The ranker operates on (user_emb, item_emb) from the trained MF.
  - We do NOT use user demographics or movie genres for the main experiment.
  - Reason: adding content features would confound the sparsity comparison
    (the ranker would benefit from features, not just interaction density).
  - Possible ablation (if time permits): +genre features to the ranker only.

=== ID REMAPPING ===

UserIDs  : 1-6040  → remap to 0-6039  (contiguous, no gaps).
MovieIDs : 1-3952  → remap to 0-N-1   (N = n_items in ratings, skip gap IDs).
Both maps will be saved to data/processed/ for reproducibility.
""")

In [ ]:
# Verify the ID gap situation
movie_ids_in_ratings = sorted(ratings['movie_id'].unique())
full_range = list(range(1, ratings['movie_id'].max() + 1))
gap_ids = sorted(set(full_range) - set(movie_ids_in_ratings))
print(f'MovieID range: 1 – {ratings["movie_id"].max()}')
print(f'Unique movie IDs in ratings.dat: {len(movie_ids_in_ratings)}')
print(f'Gap IDs (in range but not in ratings): {len(gap_ids)}')
print(f'First 20 gap IDs: {gap_ids[:20]}')
print()
print('→ We remap only the IDs that appear in ratings.dat — gap IDs are simply skipped.')

## 10. Sparsity sweep sanity check

In [ ]:
# How many training interactions remain at each density level?
# (after removing 2 interactions per user for val + test)
train_pool_per_user = user_counts - 2  # val + test held out
train_pool_per_user = train_pool_per_user.clip(lower=1)

print(f'Total training interactions at each density:')
print(f'{"Density":>10} | {"Total train":>14} | {"Min/user":>10} | {"Median/user":>12} | {"Users w/ <3 train":>18}')
print('-' * 75)
for d in [1.0, 0.8, 0.6, 0.4, 0.2]:
    sampled = (train_pool_per_user * d).astype(int).clip(lower=1)
    total = sampled.sum()
    pct_lt3 = (sampled < 3).mean() * 100
    print(f'{d:>10.0%} | {total:>14,} | {sampled.min():>10} | {sampled.median():>12.0f} | {pct_lt3:>18.1f}%')

In [ ]:
# Plot the distribution of training interactions at each density level
fig, axes = plt.subplots(1, 5, figsize=(18, 3), sharey=False)
densities = [1.0, 0.8, 0.6, 0.4, 0.2]
colors = sns.color_palette('Blues_r', 5)

for ax, d, c in zip(axes, densities, colors):
    sampled = (train_pool_per_user * d).astype(int).clip(lower=1)
    ax.hist(sampled, bins=40, color=c, edgecolor='white', linewidth=0.2)
    ax.set_title(f'Density {d:.0%}')
    ax.set_xlabel('Train interactions')
    if ax == axes[0]:
        ax.set_ylabel('Users')

plt.suptitle('Training Interactions per User at Each Sparsity Level', y=1.02)
plt.tight_layout()
plt.savefig(FIG / 'eda_sparsity_distributions.png', dpi=150)
plt.show()

## 11. Summary of EDA findings and decisions

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║              EDA FINDINGS AND DECISIONS                          ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  DATASET                                                         ║
║  • 1,000,209 ratings, 6,040 users, ~3,706 unique movies           ║
║  • Matrix density ~4.5% — appropriately sparse for RecSys        ║
║  • All users have ≥ 20 ratings (per README, confirmed)           ║
║                                                                  ║
║  RATINGS                                                         ║
║  • Bimodal-ish: peaks at 3 and 4 stars                           ║
║  • We BINARIZE: any observed rating = positive interaction       ║
║  • No filtering by rating value (standard implicit CF protocol)  ║
║                                                                  ║
║  SPLIT STRATEGY                                                  ║
║  • Leave-one-out: last item by timestamp = TEST                  ║
║                   second-to-last = VAL                           ║
║  • Negatives: 99 sampled per user for eval (fixed seed=42)       ║
║  • Sparsity sweep: subsample TRAIN only, at d∈{1,0.8,0.6,0.4,0.2}║
║                                                                  ║
║  FEATURE ENGINEERING                                             ║
║  • NONE needed for MF, NCF, or two-stage ranker (core models)   ║
║  • Input = (user_id, movie_id) only, IDs remapped to 0-indexed  ║
║  • Genres/demographics excluded from main experiment             ║
║    to isolate the effect of interaction density                  ║
║                                                                  ║
║  ITEM POPULARITY                                                 ║
║  • Heavy power law: top ~20% items cover ~80% of ratings        ║
║  • Implication: negative sampling may over-sample cold items;   ║
║    popular items dominate retrieval candidates for MF            ║
║                                                                  ║
║  NEXT STEP                                                       ║
║  • Implement data.py: load, remap IDs, split, negative sample   ║
║  • Validate: reconstruct ratings from split → count matches       ║
╚══════════════════════════════════════════════════════════════════╝
""")